# 03. 딥러닝 시계열 모델의 발전 과정 — RNN에서 PatchTST까지

> **Day 01 — 제조 시계열 AI**
> - Transformer가 제안되기까지 시계열 데이터를 분석하기 위한 딥러닝 방법론 흐름 이해하기
> - 항공기 엔진의 잔여수명(RUL)을 예측하며 RNN → Attention → Transformer → PatchTST로 이어지는 발전 과정을 직접 구현한다.

> **실습 안내**
> `"""채워넣기"""` 가 적힌 셀은 코드 대부분이 이미 있고, `"""채워넣기"""`로 표시된 부분만 채우면 됩니다.
> 나머지 셀은 실행 결과를 확인하며 따라오시면 됩니다.
> 막히는 부분은 손을 들어 주세요.

## 목차

**[이론]**

1. 문제 설정 — 항공기 엔진 RUL 예측
2. RNN/LSTM — 순차 처리 구조와 그 한계
3. LSTM 베이스라인 구축 — 비교의 기준
4. Seq2Seq의 구조적 병목 — 고정 길이 컨텍스트의 한계
5. Attention 메커니즘의 도입과 구현
6. Transformer Encoder의 구성
7. 시계열 데이터 적용의 한계


**[실습]**

8. PatchTST의 두 아이디어 — Patching · Channel Independence
9. PatchTST 템플릿 채워넣기와 학습
10. LSTM vs PatchTST — 발전 과정을 숫자로 정리하기

In [ ]:
# 공통 준비 — Colab / 로컬 양쪽에서 동작
import os, random, warnings
import numpy as np
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print("PyTorch 미설치 — 다음 셀에서 설치합니다.")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print(f"Colab 환경: {IN_COLAB}")

In [ ]:
# ══ 실습 자료 위치 — 강사가 배포한 주소로 이 한 줄만 맞추면 됩니다 ══════════
DATA_REPO = "https://raw.githubusercontent.com/leejiyoon52/ai-course/main/day1"

# Google Drive로 배포받았다면, 위 줄 대신 아래 두 줄의 주석을 푸십시오.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_REPO = "file:///content/drive/MyDrive/ai-course/day1"
# ═══════════════════════════════════════════════════════════════════════

import os, shutil, urllib.request
os.environ["MFG_DATA_BASE"] = DATA_REPO          # loaders.py가 이 값을 읽습니다

for f in ["mfg_datagen.py", "loaders.py"]:
    if os.path.exists(f):
        continue
    src = f"{DATA_REPO}/modules/{f}"
    try:
        if src.startswith("file://"):
            shutil.copy(src[len("file://"):], f)
        else:
            urllib.request.urlretrieve(src, f)
        print(f"다운로드 완료: {f}")
    except Exception:
        print(f"⚠️ {f} 를 받지 못했습니다 — 왼쪽 파일 탭에 직접 업로드해 주세요")

import mfg_datagen
import loaders
print(f"실습 모듈 준비 완료 — 자료 위치: {DATA_REPO}")

---
## 1. 문제 설정 — 항공기 엔진 RUL 예측

**RUL(Remaining Useful Life, 잔여 유효 수명)** 은 설비가 고장까지 몇 사이클을
더 버틸 수 있는가를 나타내는 지표입니다. 직관적으로는 설비의 *남은 체력 게이지*에 해당합니다.

- 데이터: NASA **C-MAPSS** — 항공기 터보팬 엔진 열화 시뮬레이터. 엔진 100대가
  정상 가동에서 고장까지 이르는 전체 궤적을 **21개 센서**로 기록했습니다.
- 문제의 의의: RUL 예측은 정비 시점을 사전에 계획할 수 있게 합니다 — 사후 정비에서
  예지보전으로 전환하는 것으로, NB02에서 다룬 예지보전 개념과 동일한 문제입니다.
- 제조 현장의 동일 문제: CNC 공구 수명, 펌프·컴프레서 베어링 수명, 배터리 셀 용량 열화.

**데이터 설명(Data Description)**

| 항목 | 내용 |
|---|---|
| 데이터셋 | NASA C-MAPSS FD001 (실데이터) |
| 규모 | 엔진(unit) 100대 · train 약 20,631행 |
| 단위 | cycle — 시간이 아니라 가동 사이클 수 기준 |
| 컬럼 | `unit`, `cycle`, 운전조건 `op1~op3`, 센서 `s1~s21` (총 26개) |
| 라벨 | RUL(잔여 사이클) — train은 각 엔진의 정상 가동부터 고장까지 전 구간을 포함 |

---
| 컬럼 | 의미 |
|---|---|
| `unit` | 엔진 번호(1~100) — 어느 엔진의 기록인지 구분하는 ID |
| `cycle` | 가동 사이클 번호 — 이 데이터의 "시간" 역할. 같은 `unit` 안에서 값이 커질수록 열화가 진행된 것 |
| `op1~op3` | 운전 조건(고도·마하수·스로틀 각도) — 이 사이클에 엔진이 어떤 환경에서 돌았는가 |
| `s1~s21` | 센서 측정값 21개 — 온도·압력·회전수·연료 유량 등 실제 물리 측정 결과 |

`unit`+`cycle`이 이 데이터의 좌표계이고, 운전조건(입력 환경)과 센서(측정 결과)는 성격이 다릅니다.
FD001은 운전조건이 거의 고정되어 있지만, NB05에서 다룰 FD003은 이 값이 흔들립니다.
센서 21개 중 열화 추세가 뚜렷한 8개만 골라 쓰는 이유는 잠시 뒤 확인합니다.

FD001은 단일 운전조건·단일 고장모드(HPC 열화)로 구성된 가장 단순한 서브셋입니다.

In [ ]:
# C-MAPSS FD001 로드 — 실데이터 우선, 실패 시 합성 폴백
"""채워넣기"""
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 3.5)
plt.rcParams["axes.grid"] = True

train_raw, test_raw, test_rul = loaders.load_cmapss("""채워넣기""")
print(f"train: {train_raw.shape} | 엔진 수: {train_raw['unit'].nunique()}")
print(f"컬럼: unit, cycle, 운전조건 op1~3, 센서 s1~s21")
train_raw.head(3)

In [ ]:
# 엔진 한 대의 일생 — 열화가 센서에 어떻게 새겨지는가
"""채워넣기"""
u1 = train_raw["""채워넣기"""]
sensors_peek = ["s2", "s4", "s11", "s15"]

fig, axes = plt.subplots(2, 2, figsize=(12, 5))
for ax, s in zip(axes.ravel(), sensors_peek):
    ax.plot(u1["cycle"], u1[s], lw=0.8)
    ax.set_title(f"unit 1 — {s}")
    ax.set_xlabel("cycle")
plt.tight_layout()
plt.show()
print(f"unit 1의 수명: {u1['cycle'].max()}사이클 — 마지막 사이클이 곧 고장 시점입니다.")
print("고장이 가까워질수록 여러 센서가 함께 추세를 그립니다. 이 추세를 읽는 것이 RUL 예측입니다.")

In [ ]:
# 엔진마다 수명이 얼마나 다른가 — 수명 분포
lifes = train_raw.groupby("unit")["cycle"].max()
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(lifes, bins=25)
ax.set_xlabel("engine life (cycles)"); ax.set_ylabel("count")
ax.set_title("Engine lifetime distribution")
plt.tight_layout(); plt.show()
print(f"수명 범위: {lifes.min()} ~ {lifes.max()}사이클 (평균 {lifes.mean():.0f})")
print("같은 기종인데 수명이 2배 넘게 차이 납니다 — 평균 수명 교체(TBM)가 낭비이거나 위험한 이유입니다.")

In [ ]:
# 잔여수명 (RUL) 라벨 만들기 — 고장까지 남은 사이클, 상한 125로 클리핑
"""채워넣기"""
train = loaders.add_rul(train_raw, cap="""채워넣기""")

u1 = train[train["unit"] == 1]
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(u1["cycle"], u1["RUL"], lw=1.5)
ax.set_xlabel("cycle"); ax.set_ylabel("RUL")
ax.set_title("Piecewise-linear RUL label (cap=125)")
plt.tight_layout()
plt.show()
print("초반 구간은 열화가 없어 '정확한 남은 수명'을 알 수 없으므로 관례상 일정값(125)으로 눌러 둡니다.")

---
## 2. RNN/LSTM — 순차 처리 구조와 그 한계

RNN(Recurrent Neural Network, 순환 신경망)은 시점 순서대로 hidden state를 갱신하는 구조입니다.
갱신이 반복될수록 초반 시점의 정보는 점차 희석되어, 먼 과거의 정보가 현재 시점까지 온전히 전달되지 않습니다.

- **장기 의존성 소실**: 수백 스텝 이전의 정보는 반복적인 갱신 과정에서 점차 소실됩니다.
- **순차 연산 병목**: t 시점의 계산은 t-1 시점이 완료되어야 시작할 수 있어, GPU의 병렬 연산 이점을 활용하지 못합니다.

LSTM(Long Short-Term Memory)은 게이트 구조를 통해 정보 소실을 완화한 개선 모델입니다.
다만 완전한 해결이 아니라 **완화**라는 점이 이후 논의의 출발점입니다.

In [ ]:
# 정보가 소실되는 속도 — 갱신마다 일정 비율만 남아도 반복되면 곱셈으로 줄어든다
import math
import torch

steps = torch.arange(0, 300)
for keep in [0.99, 0.95, 0.90]:
    plt.plot(steps, keep ** steps, label=f"keep {keep}")
plt.axhline(0.01, color="red", ls="--", lw=0.8)
plt.title("How much of step-0 information survives after t steps")
plt.xlabel("steps later"); plt.ylabel("surviving fraction")
plt.legend(); plt.tight_layout(); plt.show()

for keep in [0.99, 0.95, 0.90]:
    t_1pct = math.log(0.01) / math.log(keep)
    print(f"keep={keep} (스텝당 {100 * (1 - keep):.0f}% 손실) → 1% 아래로 떨어지는 시점: 약 {t_1pct:.0f}스텝")
print("\n스텝당 손실이 작아도(1~10%) 반복해서 곱해지므로, 정보는 선형이 아니라 기하급수적으로 감소합니다.")
print("300사이클 엔진 이력의 초반 징후는 마지막 hidden state에 거의 남지 않습니다.")

In [ ]:
# 입력 준비 ① — 21개 센서 중 추세가 뚜렷한 8개만 선택, train 통계로만 스케일링
"""채워넣기"""
from sklearn.preprocessing import StandardScaler

ALL_SENSORS = [f"s{i}" for i in range(1, 22)]
constant = [s for s in ALL_SENSORS if """채워넣기"""]
print(f"완전히 일정한(값이 변하지 않는) 센서 {len(constant)}개: {constant}")
print("→ 값 자체가 열화와 무관하게 고정돼 있어 정보가 없는 컬럼입니다.")

SENSORS = ["s2", "s3", "s4", "s7", "s11", "s12", "s15", "s21"]
print(f"\n이 중 열화 추세가 뚜렷한 {len(SENSORS)}개를 선택해 사용합니다: {SENSORS}")
print("(C-MAPSS 연구에서 흔히 쓰는 서브셋입니다 — 나머지 센서도 변화는 있지만 이번 실습에서는 다루지 않습니다)")

units = train["unit"].unique()
n_tr = int(len(units) * 0.8)
tr_units, te_units = units[:n_tr], units[n_tr:]           # 유닛 단위 분할 (누수 방지)

scaler = StandardScaler().fit(train.loc[train["unit"].isin(tr_units), SENSORS])
train[SENSORS] = scaler.transform(train[SENSORS])
print(f"\n학습 엔진 {len(tr_units)}대 / 평가 엔진 {len(te_units)}대")
print("NB02 원칙 그대로 — 스케일러는 학습 유닛 통계로만 fit 했습니다.")

In [ ]:
# 입력 준비 ② — 유닛별 슬라이딩 윈도우 (NB02 make_windows의 다변량 버전)
"""채워넣기"""
WIN = 30

def make_unit_windows(df, units, sensors, win=WIN, stride=2):
    """
    X: (윈도우 수, win, len(sensors)) — 특정 엔진의 연속된 win사이클 동안의 센서값
    y: (윈도우 수,) — 그 윈도우가 끝나는 시점의 RUL(잔여 사이클) 스칼라값
    "최근 win사이클의 센서 흐름을 보고 그 시점의 잔여수명(RUL)을 맞히는" 회귀(Regression) 문제입니다.
    """
    X, y = [], []
    for u in units:
        g = df[df["unit"] == u]
        arr, rul = g[sensors].values, g["RUL"].values
        for i in range(0, len(g) - win, stride):
            X.append(arr["""채워넣기"""])                     # X: (win, 센서 수) 윈도우
            y.append(rul["""채워넣기"""])                      # y: 윈도우 마지막 시점의 RUL(스칼라)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_tr, y_tr = make_unit_windows(train, tr_units, SENSORS)
X_te, y_te = make_unit_windows(train, te_units, SENSORS, stride=1)
print(f"X_tr {X_tr.shape} (윈도우, 시점, 센서) | X_te {X_te.shape}")
print(f"y_tr {y_tr.shape} — 윈도우 하나당 RUL 값 하나(스칼라) → 회귀(Regression) 문제입니다")

In [ ]:
# PyTorch DataLoader 준비
"""채워넣기"""
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RUL_CAP = 125.0

def to_loader(X, y, batch=256, shuffle=True):
    ds = TensorDataset(torch.tensor(X), torch.tensor("""채워넣기"""))   # 타깃을 0~1로 정규화
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)

dl_tr = to_loader(X_tr, y_tr)
dl_te = to_loader(X_te, y_te, shuffle=False)
print(f"배치 수: train {len(dl_tr)} / test {len(dl_te)} (batch=256)")
print("※ 타깃(0~125)을 0~1로 정규화했습니다 — 스케일 큰 타깃은 수렴을 크게 늦춥니다.")
print("  (초기 예측은 0 근처인데 타깃이 크면 오차·그래디언트가 커져 학습이 불안정해집니다)")
print("※ DataLoader의 shuffle은 '윈도우 순서'만 섞습니다 — 분할이 끝난 뒤라 누수가 아닙니다.")

---
## 3. LSTM 베이스라인 구축 — 비교의 기준

이후 모든 모델의 성능은 이 기준선과 비교하여 평가합니다. 기준선 없는 성능 주장은 근거가 부족합니다.

In [ ]:
# LSTM RUL 모델 정의 — 마지막 hidden state로 회귀
"""채워넣기"""
class LSTMRul(nn.Module):
    def __init__(self, n_sensors, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):                  # x: (B, L, C)
        out, _ = self.lstm(x)
        return self.head(out["""채워넣기"""]).squeeze(-1)   # 마지막 시점의 기억으로 예측

lstm = LSTMRul(len(SENSORS)).to(DEVICE)
n_params_lstm = sum(p.numel() for p in lstm.parameters())
print(lstm)
print(f"파라미터 수: {n_params_lstm:,}")

In [ ]:
# LSTM 학습 — 5 epoch 데모
"""채워넣기"""
# 학습 약 20초 소요 (T4 기준)
import time

def train_model(model, dl, epochs=5, lr=3e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = """채워넣기"""
            loss.backward(); opt.step()
            tot += loss.item() * len(xb)
        rmse_c = (tot / len(dl.dataset)) ** 0.5 * RUL_CAP
        print(f"epoch {ep + 1}/{epochs} | train RMSE ≈ {rmse_c:5.1f} cycles | {time.time() - t0:.0f}s")
    return time.time() - t0

t_lstm = train_model(lstm, dl_tr)

In [ ]:
# 기준선 확정 — 평가 엔진에서 MAE·RMSE
"""채워넣기"""
def evaluate(model, dl):
    model.eval(); preds, ys = [], []
    with torch.no_grad():
        for xb, yb in dl:
            preds.append(model(xb.to(DEVICE)).cpu().numpy()); ys.append(yb.numpy())
    p, t = np.concatenate(preds) * RUL_CAP, np.concatenate(ys) * RUL_CAP   # 사이클 단위 복원
    return p, t, """채워넣기""", """채워넣기"""

pred_l, true_l, mae_lstm, rmse_lstm = evaluate(lstm, dl_te)
print(f"LSTM 기준선 → MAE {mae_lstm:.2f} / RMSE {rmse_lstm:.2f} 사이클")
print("이 값이 이후 PatchTST와 비교할 기준선입니다.")

In [ ]:
# 예측 vs 실제 산점도 — 오차가 어디에 몰리는지 본다
fig, ax = plt.subplots(figsize=(4.5, 4.2))
ax.scatter(true_l, pred_l, s=4, alpha=0.3)
ax.plot([0, 125], [0, 125], "r--", lw=1)
ax.set_xlabel("actual RUL"); ax.set_ylabel("predicted RUL")
ax.set_title("LSTM baseline")
plt.tight_layout(); plt.show()
print("빨간 대각선이 완벽한 예측입니다. 대각선에서 세로로 떨어진 거리가 오차입니다.")

---
## 4. Seq2Seq의 구조적 병목 — 고정 길이 컨텍스트의 한계

번역을 위해 설계된 Seq2Seq 구조는 인코더가 입력 전체를 **고정 길이 컨텍스트 벡터
하나**로 압축해 디코더에 전달합니다.

```
[x1 x2 x3 ... x300]  →  인코더  →  [ 벡터 1개 ]  →  디코더  →  출력
                                   ↑ 정보 손실 발생 지점
```

- 입력이 길어질수록 벡터 하나에 압축되며 손실되는 정보가 커집니다.
- 300사이클 엔진 이력의 초반 이상 징후는 이 압축 과정에서 소실됩니다.
- 이 병목에 대한 해법으로 제안된 것이 Attention이며, 다음 절에서 직접 구현합니다.

In [ ]:
# 고정 길이 압축 실험 — 서로 다른 두 이력을 벡터 하나로 압축하면 구별이 사라진다
t = np.linspace(0, 1, 300)
normal = 0.3 * np.sin(2 * np.pi * 12 * t) + t          # 정상 열화 이력
spiked = normal.copy(); spiked[20:24] += 2.5           # 초반에 이상 스파이크가 있던 이력

note1, note2 = normal.mean(), spiked.mean()            # 고정 길이 압축 = 벡터 1개(평균)로 요약
print(f"정상 이력 요약값   : {note1:.4f}")
print(f"스파이크 이력 요약값: {note2:.4f}")
print(f"차이               : {abs(note1 - note2):.4f} — 300스텝 중 4스텝의 사건은 요약 과정에서 소실됩니다")
print("압축 이후에는 초반의 이상 징후를 복구할 수 없습니다. 원본을 참조하는 방식이 필요합니다.")

---
## 5. Attention 메커니즘의 도입과 구현

Attention은 입력을 고정 벡터로 압축하지 않고, 필요한 시점의 정보를 그때그때 참조하는 방식입니다.

**Scaled Dot-Product Attention** 수식은 한 줄입니다.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

- **Q(query)**: 지금 내가 찾고 싶은 것 / **K(key)**: 각 시점의 색인 / **V(value)**: 각 시점의 내용
- $QK^\top$: 모든 시점 쌍의 관련도 점수 / $\sqrt{d_k}$: 점수 폭주 방지 / softmax: 확률로 정규화
- 수식 그대로 코드 5줄입니다. 아래에서 직접 씁니다.

In [ ]:
# Self-Attention을 코드 5줄로 — 수식과 한 줄씩 대응시킨다
"""채워넣기"""
def self_attention(x, Wq, Wk, Wv):
    # x: (B, L, D) — 배치 크기 B, 시점 수(시퀀스 길이) L, 시점별 표현 벡터 차원 D
    Q, K, V = x @ Wq, x @ Wk, x @ Wv                     # 1) Q, K, V 만들기
    scores = """채워넣기"""                     # 2) 모든 시점 쌍의 관련도
    scores = scores / ("""채워넣기""")               # 3) sqrt(d_k)로 스케일
    A = torch.softmax(scores, dim=-1)                    # 4) 확률로 정규화
    return A @ V, A                                      # 5) 가중합 = 문맥 반영 표현

torch.manual_seed(SEED)
# 실제 센서 데이터가 아니라 개념 확인용 무작위 텐서입니다.
# L=6(시점 6개)은 이후 6x6 Attention 행렬을 그림으로 확인하기 좋도록 고른 값이고,
# D=8(차원 8)은 시점 하나를 표현하는 벡터 길이로, 이 데모에서는 임의로 정한 크기입니다.
x = torch.randn(1, 6, 8)                                 # (B=1, L=6, D=8)
Wq, Wk, Wv = (torch.randn(8, 8) * 0.3 for _ in range(3))
out, A = self_attention(x, Wq, Wk, Wv)
print(f"입력 {tuple(x.shape)} (B, L, D) → 출력 {tuple(out.shape)} | Attention 행렬 {tuple(A.shape)} (L x L)")

In [ ]:
# √d_k 스케일링이 왜 필요한가 — 차원이 커지면 softmax가 한 점에 몰린다
for d in [8, 64, 512]:
    q, k = torch.randn(1, d), torch.randn(6, d)
    raw = (q @ k.T).squeeze()
    sm_raw = torch.softmax(raw, dim=-1)
    sm_scaled = torch.softmax(raw / d ** 0.5, dim=-1)
    print(f"d_k={d:4d} | 스케일 없음: max {sm_raw.max():.3f} | 스케일 적용: max {sm_scaled.max():.3f}")
print("\n스케일이 없으면 고차원에서 한 시점에 확률이 쏠려, 나머지 시점의 기울기가 죽습니다.")

In [ ]:
# Attention 행렬을 그림으로 — "누가 누구를 보는가"
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(A[0].detach().numpy(), cmap="viridis")
ax.set_xlabel("attends to (key)"); ax.set_ylabel("query")
ax.set_title("Attention weights (6x6)")
plt.colorbar(im)
plt.tight_layout()
plt.show()
print("행마다 합이 1인 확률 분포입니다. 각 시점이 어느 시점을 참조했는지가 그대로 보입니다.")
print("→ NB04에서는 바로 이 행렬이 '고장 원인 추적(XAI)'의 재료가 됩니다.")

---
## 6. Transformer Encoder의 구성

Attention만으로는 두 가지가 부족합니다.

1. **순서 정보가 없다** — Attention은 집합 연산이라 시점을 섞어도 결과가 같습니다.
   → **Positional Encoding(위치 부호화)** 을 더해 순서를 새깁니다.
2. **한 종류의 관계만 본다** — 열화 추세와 사이클 리듬은 다른 관점이 필요합니다.
   → **Multi-Head**: 여러 Attention을 병렬로 돌려 서로 다른 관계를 보게 합니다.

`nn.Transformer` 같은 완제품은 쓰지 않습니다. 구성 요소를 직접 조립해야 NB04·NB05에서
내부를 열어 볼 수 있습니다.

In [ ]:
# Positional Encoding — 사인·코사인으로 위치를 새긴다
"""채워넣기"""
def positional_encoding(L, d_model):
    pos = torch.arange(L).unsqueeze(1).float()
    i = torch.arange(0, d_model, 2).float()
    angle = """채워넣기"""
    pe = torch.zeros(L, d_model)
    pe[:, 0::2] = torch.sin(angle)
    pe[:, 1::2] = torch.cos(angle)
    return pe

pe = positional_encoding(60, 32)
fig, ax = plt.subplots(figsize=(9, 3))
im = ax.imshow(pe.T, aspect="auto", cmap="RdBu")
ax.set_xlabel("position"); ax.set_ylabel("dim")
ax.set_title("Positional encoding pattern")
plt.colorbar(im)
plt.tight_layout(); plt.show()
print("Self-Attention은 시점을 동시에 처리해 순서 정보가 없으므로, 입력에 위치 정보를 명시적으로 더해줘야 합니다.")
print(f"차원(dim)마다 다른 주기의 sin/cos을 사용합니다 — 낮은 dim은 빠르게, 높은 dim은 느리게 진동해 위치({pe.shape[0]}개)마다 고유한 {pe.shape[1]}차원 벡터가 만들어집니다.")

In [ ]:
# Multi-Head Attention 직접 구현 — 여러 관점을 병렬로
"""채워넣기"""
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.dk = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)       # Q·K·V를 한 번에
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):                                # x: (B, L, D)
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, L, self.h, self.dk).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)           # (B, h, L, dk)
        A = torch.softmax("""채워넣기""", dim=-1)
        z = (A @ v).transpose(1, 2).reshape(B, L, D)     # head들을 다시 이어붙임
        return self.out(z), A

mha = MultiHeadAttention(32, n_heads=4)
z, A = mha(torch.randn(2, 60, 32))
print(f"출력 {tuple(z.shape)} | Attention {tuple(A.shape)} — head 4개가 각자 60x60 관계를 봅니다")

In [ ]:
# head마다 다른 관점 — 4개 head의 Attention 지도를 나란히
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for h, ax in enumerate(axes):
    ax.imshow(A[0, h].detach().numpy(), cmap="viridis")
    ax.set_title(f"head {h}")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Same input, four different relation maps")
plt.tight_layout(); plt.show()
print("학습 전이라 무늬는 무작위지만, 학습이 되면 head마다 추세·주기 등 다른 관계를 분담합니다.")

In [ ]:
# Encoder Block 조립 — Attention + FFN + 잔차 + 정규화
"""채워넣기"""
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                 nn.Linear(d_ff, d_model))
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)

    def forward(self, x):
        a, A = self.attn(x)
        x = self.ln1("""채워넣기""")                              # 잔차 연결 + 정규화
        x = self.ln2(x + self.ffn(x))
        return x, A

blk = EncoderBlock(32, 4, 64)
z, _ = blk(torch.randn(2, 60, 32))
print(f"블록 통과: {tuple(z.shape)} — 이 블록을 쌓으면 Transformer Encoder입니다")
print("순환이 없습니다. 60개 시점이 한 번에, 병렬로 처리됩니다.")

## 7. 시계열 데이터 적용의 한계

Transformer를 시계열에 그대로 적용하면 두 가지 한계에 직면합니다.

1. **점 하나는 단어 하나가 아니다** — 자연어에서 토큰(단어) 하나는 그 자체로 의미 단위를 이룹니다.
   반면 시계열의 한 시점 값(예: 진동 센서의 0.1초 샘플)은 노이즈에 가깝고 개별적으로는 정보량이 적습니다.
   의미 있는 패턴은 여러 시점이 모인 **구간(파형 조각)** 단위에서 드러나므로, 시점 하나를 토큰으로 다루는
   것보다 여러 시점을 묶은 구간을 토큰으로 다루는 편이 유리합니다.
2. **계산량 O(L²)** — Self-Attention은 모든 시점 쌍의 관련도를 계산하므로, Attention 행렬 크기와 계산량이
   시퀀스 길이 L의 제곱에 비례합니다. 자연어 문장은 보통 수백 토큰 수준이지만, 시계열은 샘플링 주기에 따라
   한 윈도우가 수천 스텝에 이를 수 있어 L이 커질수록 계산 비용이 급격히 늘어납니다.

이 두 한계는 이어지는 PatchTST의 두 아이디어(Patching, Channel Independence)와 직접 연결됩니다 — 시점을
구간(patch)으로 묶으면 ①의 의미 단위 문제를 완화하는 동시에, 유효 시퀀스 길이를 줄여 ②의 계산량도 낮출 수 있습니다.

In [ ]:
# O(L²)의 영향 확인 — 길이를 늘리며 Attention 행렬 크기와 시간을 측정한다
import time

for L in [100, 400, 1600]:
    x = torch.randn(1, L, 32)
    t0 = time.time()
    _ = mha(x)
    dt = (time.time() - t0) * 1000
    print(f"L={L:5d} | Attention 행렬 {L}x{L} = {L * L:>9,}칸 | {dt:6.1f} ms")
print("\n길이 4배 → 행렬 16배. 1초 샘플링 하루치(86,400스텝)는 이대로면 불가능합니다.")
print("→ PatchTST는 이 두 문제를 함께 해결합니다.")

In [ ]:
# 실습부 재설정 — 이 셀부터 실행해도 오후 실습이 완주되도록 전부 다시 준비합니다
import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (11, 3.5); plt.rcParams["axes.grid"] = True

import loaders
print(f"재설정 완료 | device: {DEVICE}")

In [ ]:
# 데이터 재준비
train_raw, _, _ = loaders.load_cmapss("FD001")
train = loaders.add_rul(train_raw, cap=125)
SENSORS = ["s2", "s3", "s4", "s7", "s11", "s12", "s15", "s21"]
WIN = 30

units = train["unit"].unique()
n_tr = int(len(units) * 0.8)
tr_units, te_units = units[:n_tr], units[n_tr:]
scaler = StandardScaler().fit(train.loc[train["unit"].isin(tr_units), SENSORS])
train[SENSORS] = scaler.transform(train[SENSORS])

def make_unit_windows(df, units, sensors, win=WIN, stride=2):
    X, y = [], []
    for u in units:
        g = df[df["unit"] == u]
        arr, rul = g[sensors].values, g["RUL"].values
        for i in range(0, len(g) - win, stride):
            X.append(arr[i : i + win]); y.append(rul[i + win - 1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_tr, y_tr = make_unit_windows(train, tr_units, SENSORS)
X_te, y_te = make_unit_windows(train, te_units, SENSORS, stride=1)
RUL_CAP = 125.0
dl_tr = DataLoader(TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr / RUL_CAP)),
                   batch_size=256, shuffle=True)
dl_te = DataLoader(TensorDataset(torch.tensor(X_te), torch.tensor(y_te / RUL_CAP)),
                   batch_size=256, shuffle=False)
print(f"X_tr {X_tr.shape} | X_te {X_te.shape} (타깃은 0~1 정규화)")

---
## 8. PatchTST의 두 아이디어 — Patching · Channel Independence

**① Patching** — 파형을 초 단위가 아니라 사이클 단위로 처리

- 시계열을 점이 아니라 **패치(작은 구간)** 단위로 묶어 하나의 토큰으로 만듭니다.
- 점 하나는 무의미해도 패치 하나에는 파형의 의미가 담깁니다.
- 시퀀스 길이가 패치 수로 줄어들어 **O(L²) 계산량도 함께 준다**는 것이 핵심입니다.

$$L\ \text{시점} \;\xrightarrow{\;\text{길이 } P,\ \text{보폭 } S\;}\; N = \left\lfloor \frac{L - P}{S} \right\rfloor + 1\ \text{패치}$$

**② Channel Independence** — 센서(채널)마다 독립적으로 처리

- 센서(채널)마다 파형의 성격이 다릅니다. 억지로 섞어 넣는 대신 **채널별로 독립 처리**하고
  가중치만 공유합니다.
- 제조 다변량 센서(온도·압력·진동이 함께 존재하는)에 직결되는 설계입니다.

In [ ]:
# Patch 분할 구현 — NB02의 슬라이딩 윈도우를 윈도우 '안'에 또 적용
"""채워넣기"""
PATCH, STRIDE = 10, 5  # PATCH: 패치 하나의 길이(시점 수), STRIDE: 패치를 자르는 보폭

def patchify(x, patch=PATCH, stride=STRIDE):
    """(B, L, C) → (B, C, N, P): 채널(센서)별로 길이 P 패치 N개
    B: 배치(윈도우 수), L: 원래 시퀀스 길이(WIN), C: 채널(센서) 수, N: 패치 개수, P: 패치 길이"""
    x = x.transpose(1, 2)                                # (B, L, C) → (B, C, L)
    return x.unfold("""채워넣기""")  # L 축을 patch 길이로 잘라 (B, C, N, P)

xb = torch.tensor(X_tr[:4])                              # B=4개 윈도우, L=WIN 시점, C=센서 수
patches = patchify(xb)
B, C, N, P = patches.shape
print(f"윈도우 {tuple(xb.shape)} (B, L, C) → 패치 {tuple(patches.shape)} (B, C, N, P)")
print(f"B={B}(윈도우 수) C={C}(센서 채널 수) N={N}(패치 개수) P={P}(패치 길이, 시점 {P}개씩 묶음)")
print(f"시퀀스 길이가 {xb.shape[1]} → {N} 으로 줄었습니다. Attention 행렬은 {xb.shape[1]}² → {N}²")

In [ ]:
# 패치가 파형을 어떻게 써는지 눈으로 확인
sig = X_tr[0, :, 0]
starts = list(range(0, len(sig) - PATCH + 1, STRIDE))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})
ax1.plot(sig, lw=1, color="gray", label="window (s2)")
ax1.set_title(f"One window sliced into patches (P={PATCH}, S={STRIDE})")
ax1.legend()

# 패치가 겹치므로(STRIDE < PATCH) 한 줄에 모두 그리면 서로 가려집니다 — 패치마다 행(row)을 따로 둡니다
for row, j in enumerate(starts):
    ax2.barh(row, PATCH, left=j, height=0.8, alpha=0.5)
ax2.set_yticks(range(len(starts)))
ax2.set_yticklabels([f"patch {i}" for i in range(len(starts))])
ax2.set_xlabel("time step")
ax2.invert_yaxis()
plt.tight_layout(); plt.show()
print(f"패치 {len(starts)}개가 STRIDE={STRIDE}만큼씩 겹치며 미끄러집니다 — 행(row) 하나하나가 '토큰' 하나입니다.")

---
## 9. PatchTST 템플릿 채워넣기와 학습

아래 두 구성 요소를 직접 구현합니다. 완성된 모델 코드는 이후 별도로 제공되므로,
구현 중 막히는 부분이 있어도 실습을 이어갈 수 있으며, 각 단계는 출력 shape 검증으로 정확성을 확인합니다.

| 직접구현 | 구현 내용 | 검증 방법 |
|---|---|---|
| ① Patch 임베딩 | 패치(P차원)를 모델 차원(d_model)으로 투영하고 위치 부호화를 더함 | 출력 shape |
| ② Encoder 구성 | EncoderBlock을 n_layers만큼 쌓음 | 출력 shape |

In [ ]:
def positional_encoding(L, d_model):
    pos = torch.arange(L).unsqueeze(1).float()
    i = torch.arange(0, d_model, 2).float()
    angle = pos / (10000 ** (i / d_model))
    pe = torch.zeros(L, d_model)
    pe[:, 0::2] = torch.sin(angle); pe[:, 1::2] = torch.cos(angle)
    return pe

class MultiHeadAttention(nn.Module):
    """Self-Attention을 n_heads개로 나눠 병렬 수행 — head마다 서로 다른 관점의 관련도를 학습"""
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h, self.dk = n_heads, d_model // n_heads   # h: head 수, dk: head 하나의 차원(d_model/h)
        self.qkv = nn.Linear(d_model, 3 * d_model)       # Q, K, V를 한 번에 투영
        self.out = nn.Linear(d_model, d_model)           # head별 결과를 이어붙인 뒤 다시 투영
    def forward(self, x):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, L, self.h, self.dk).transpose(1, 2)  # (B, L, D) → (B, h, L, dk)
        q, k, v = split(q), split(k), split(v)
        A = torch.softmax(q @ k.transpose(-2, -1) / self.dk ** 0.5, dim=-1)
        return self.out((A @ v).transpose(1, 2).reshape(B, L, D)), A     # head 결과를 다시 (B, L, D)로

class EncoderBlock(nn.Module):
    """Transformer Encoder 한 층 — Self-Attention 서브층과 FFN 서브층 각각에 잔차 연결 + LayerNorm"""
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
    def forward(self, x):
        a, A = self.attn(x)
        x = self.ln1(x + a)                  # 잔차 연결(residual) + LayerNorm
        return self.ln2(x + self.ffn(x)), A  # FFN 서브층도 동일하게 잔차 연결 + LayerNorm

print("구성 요소 준비 완료: positional_encoding / MultiHeadAttention / EncoderBlock")

In [ ]:
# Patch 임베딩부 — 패치를 d_model 차원 토큰으로 만든다
"""채워넣기"""
def embed_patches(patches, proj, d_model=32):
    """patches: (B, C, N, P) → (B*C, N, d_model)  ※ 채널 독립: 채널을 배치로 접는다"""
    B, C, N, P = patches.shape
    tokens = patches.reshape("""채워넣기""")        # 채널을 배치 차원으로 (Channel Independence)
    tokens = proj(tokens)                        # 선형 투영: P → d_model
    tokens = tokens + positional_encoding(N, d_model).to(tokens.device)  # 위치 부호화
    return tokens

proj = nn.Linear(PATCH, 32)
tok = embed_patches(patches, proj)
print(f"패치 {tuple(patches.shape)} → 토큰 {tuple(tok.shape)}")
assert tok.shape == (B * C, N, 32), "shape이 다르면 다시 확인해 보세요"
print("통과 ✓ — 채널 8개가 각각 독립된 시퀀스로 인코더에 들어갈 준비가 되었습니다")

In [ ]:
# Encoder 구성부 — 블록을 n_layers만큼 쌓아 통과시킨다
"""채워넣기"""
def build_encoder(d_model=32, n_heads=4, d_ff=64, n_layers=2):
    return nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range("""채워넣기""")])

def encode(tokens, blocks):
    A_last = None
    for blk in blocks:
        tokens, A_last = """채워넣기"""             # 블록을 차례로 통과
    return tokens, A_last

blocks = build_encoder()
z, A_last = encode(tok, blocks)
print(f"인코딩 결과 {tuple(z.shape)} | 마지막 층 Attention {tuple(A_last.shape)}")
assert z.shape == tok.shape, "shape이 다르면 다시 확인해 보세요"
print("통과 ✓ — 이 두 연습이 그대로 아래 완성 모델의 내부입니다")

In [ ]:
# PatchTST 조립 — 완성 코드 (위 연습과 동일한 구조)
"""채워넣기"""
class PatchTST(nn.Module):
    def __init__(self, n_sensors, win=WIN, patch=PATCH, stride=STRIDE,
                 d_model=32, n_heads=4, d_ff=64, n_layers=2):
        super().__init__()
        self.patch, self.stride = patch, stride
        self.n_patches = """채워넣기"""
        self.proj = nn.Linear(patch, d_model)
        self.register_buffer("pe", positional_encoding(self.n_patches, d_model))
        self.blocks = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.head = nn.Linear(n_sensors * d_model, 1)

    def forward(self, x):                                # x: (B, L, C)
        B, L, C = x.shape
        p = x.transpose(1, 2).unfold(2, self.patch, self.stride)   # (B, C, N, P)
        t = self.proj(p.reshape(B * C, self.n_patches, self.patch)) + self.pe
        for blk in self.blocks:
            t, A = blk(t)
        t = t.mean(dim=1).reshape(B, C * t.shape[-1])    # 패치 평균 → 채널 이어붙임
        return self.head(t).squeeze(-1)

ptst = PatchTST(len(SENSORS)).to(DEVICE)
n_params_ptst = sum(p.numel() for p in ptst.parameters())
print(f"PatchTST 파라미터: {n_params_ptst:,}")

In [ ]:
# PatchTST 학습 — LSTM과 같은 조건(5 epoch)
"""채워넣기"""
# 학습 약 30초 소요 (T4 기준)
import time

def train_model(model, dl, epochs=5, lr=3e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.MSELoss()
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tot = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = """채워넣기"""
            loss.backward(); opt.step()
            tot += loss.item() * len(xb)
        rmse_c = (tot / len(dl.dataset)) ** 0.5 * RUL_CAP
        print(f"epoch {ep + 1}/{epochs} | train RMSE ≈ {rmse_c:5.1f} cycles | {time.time() - t0:.0f}s")
    return time.time() - t0

t_ptst = train_model(ptst, dl_tr)

In [ ]:
# PatchTST 평가 — 예측 vs 실제
"""채워넣기"""
def evaluate(model, dl):
    model.eval(); preds, ys = [], []
    with torch.no_grad():
        for xb, yb in dl:
            preds.append(model(xb.to(DEVICE)).cpu().numpy()); ys.append(yb.numpy())
    p, t = np.concatenate(preds) * RUL_CAP, np.concatenate(ys) * RUL_CAP
    return p, t, """채워넣기""", """채워넣기"""

pred_p, true_p, mae_ptst, rmse_ptst = evaluate(ptst, dl_te)
print(f"PatchTST → MAE {mae_ptst:.2f} / RMSE {rmse_ptst:.2f} 사이클")

In [ ]:
# 평가 엔진 한 대의 수명 궤적 — 예측이 실제 열화를 따라가는가
u = te_units[0]
g = train[train["unit"] == u]
Xu, yu = make_unit_windows(train, [u], SENSORS, stride=1)
with torch.no_grad():
    pu = ptst(torch.tensor(Xu).to(DEVICE)).cpu().numpy() * RUL_CAP

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(yu, label="actual RUL", lw=2)
ax.plot(pu, label="PatchTST prediction", lw=1.2)
ax.set_xlabel("window index (time)"); ax.set_ylabel("RUL")
ax.set_title(f"Unit {u} — RUL trajectory")
ax.legend()
plt.tight_layout(); plt.show()
print("수명 말기(오른쪽)로 갈수록 예측이 실제에 붙는 것이 중요합니다 — 정비 판단이 걸린 구간이기 때문입니다.")

In [ ]:
# 학습된 Attention 지도 미리보기
with torch.no_grad():
    xb1 = torch.tensor(X_te[:1]).to(DEVICE)
    B1, L1, C1 = xb1.shape
    p1 = xb1.transpose(1, 2).unfold(2, ptst.patch, ptst.stride)
    t1 = ptst.proj(p1.reshape(B1 * C1, ptst.n_patches, ptst.patch)) + ptst.pe
    for blk_ in ptst.blocks:
        t1, A1 = blk_(t1)                                # A1: (B1*C1, n_heads, N, N) — 마지막 Encoder 블록의 Attention

# A1[0, 0]: 배치*채널 축의 0번째(= 테스트 샘플 1개 중 첫 센서 s2) · 헤드 0번 · 마지막 층의 (N, N) Attention 행렬
# 행(y, query patch) = 갱신 중인 패치, 열(x, attends to) = 그 패치가 참조하는 패치
# softmax가 열 방향(dim=-1)이라 각 행의 합은 1 — 한 행은 "이 패치가 N개 패치에서 정보를 가져온 비율"
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(A1[0, 0].cpu().numpy(), cmap="viridis")
ax.set_xlabel("attends to (patch)"); ax.set_ylabel("query patch")
ax.set_title("Learned attention (s2 channel, head 0)")
plt.colorbar(im); plt.tight_layout(); plt.show()
# 대각선이 밝으면 각 패치가 주로 자기 자신만 참조(지역적) / 특정 열 전체가 밝으면 그 패치가 다른 패치들의 공통 정보원 역할
# 단, s2 한 센서·헤드 하나·마지막 층만 본 단면이므로 이것만으로 "모델이 이렇게 판단했다"고 단정할 수는 없습니다
print("학습된 모델이 어느 패치를 참조해 RUL을 읽는지가 보입니다 — NB04에서는 이것으로 원인 센서를 추적합니다.")

In [ ]:
# 오전을 건너뛰고 오후만 실행한 경우 대비 — LSTM 기준선이 없으면 여기서 다시 만든다
# (오전 셀을 실행했다면 이 셀은 재사용 메시지만 출력하고 지나갑니다)
if "mae_lstm" not in dir():
    class LSTMRul(nn.Module):
        def __init__(self, n_sensors, hidden=32):
            super().__init__()
            self.lstm = nn.LSTM(n_sensors, hidden, num_layers=1, batch_first=True)
            self.head = nn.Linear(hidden, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.head(out[:, -1]).squeeze(-1)
    lstm = LSTMRul(len(SENSORS)).to(DEVICE)
    n_params_lstm = sum(p.numel() for p in lstm.parameters())
    t_lstm = train_model(lstm, dl_tr)                      # 학습 약 40초 소요 (T4 기준)
    pred_l, true_l, mae_lstm, rmse_lstm = evaluate(lstm, dl_te)
print(f"LSTM 기준선 준비 완료 — MAE {mae_lstm:.2f} / RMSE {rmse_lstm:.2f}")

In [ ]:
# 수명 구간별 오차 — 판단이 걸린 말기 구간에서 어느 모델이 정확한가
bins = [(0, 30, "late (0-30)"), (30, 80, "mid (30-80)"), (80, 126, "early (80-125)")]
rows = []
for lo, hi, name in bins:
    m = (true_p >= lo) & (true_p < hi)
    rows.append({"RUL range": name,
                 "LSTM MAE": round(np.abs(pred_l[m] - true_l[m]).mean(), 2),
                 "PatchTST MAE": round(np.abs(pred_p[m] - true_p[m]).mean(), 2),
                 "n": int(m.sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("\n평균 MAE가 같아도 말기(late) 정확도가 다르면 실무 가치는 완전히 다릅니다.")

In [ ]:
# 두 모델의 예측 산점도를 나란히
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, (p_, t_, name) in zip(axes, [(pred_l, true_l, "LSTM"), (pred_p, true_p, "PatchTST")]):
    ax.scatter(t_, p_, s=4, alpha=0.3)
    ax.plot([0, 125], [0, 125], "r--", lw=1)
    ax.set_xlabel("actual RUL"); ax.set_title(name)
axes[0].set_ylabel("predicted RUL")
plt.tight_layout(); plt.show()
print("대각선 주변에 더 촘촘히 붙는 쪽이 이긴 것입니다 — 다음 셀에서 숫자로 확정합니다.")

---
## 10. LSTM vs PatchTST — 발전 과정을 숫자로 정리하기

지금까지의 결론은 감상이 아니라 세 개의 숫자여야 합니다:
**성능, 학습 시간, 파라미터 수.**

In [ ]:
# 3축 비교표 — 오전 기준선과의 대결
"""채워넣기"""
comp = pd.DataFrame({
    "MAE (cycles)": [round(mae_lstm, 2), round(mae_ptst, 2)],
    "RMSE (cycles)": [round(rmse_lstm, 2), round(rmse_ptst, 2)],
    "train time (s)": [round(t_lstm, 1), round(t_ptst, 1)],
    "params": [f"{n_params_lstm:,}", f"{n_params_ptst:,}"],
}, index=["LSTM (baseline)", "PatchTST"])
print(comp.to_string())

better = "PatchTST" if """채워넣기""" else "LSTM"
print(f"\n이번 설정에서 MAE 우위: {better}")

**표를 읽는 법 — "최고의 모델"은 없습니다**

- PatchTST의 강점은 **긴 문맥**(윈도우를 늘릴수록)과 **다변량 채널**에서 커집니다.
  30스텝의 짧은 윈도우에서는 LSTM과 격차가 작을 수 있습니다 — 그것이 정직한 결과입니다.
- LSTM은 여전히 가볍고, 짧은 시퀀스·엣지 장비 배포에서는 합리적 선택입니다.
- 선택 기준은 항상 **데이터 길이 × 채널 수 × 배포 환경**의 삼각형입니다.

> **현장 노트**
> 논문 성능표만 보고 모델을 고르면 안 되는 이유가 이 표에 있습니다.
> 논문 벤치마크는 수백 스텝 문맥·클린 데이터 기준입니다. 우리 라인의 윈도우 길이,
> 센서 수, 추론 장비 사양으로 **직접 재본 숫자**만이 선택의 근거가 됩니다.
> PoC 단계에서 이 비교표를 우리 데이터로 다시 만드는 것이 첫 번째 할 일입니다.

> **현장 노트**
> "학습 시간"도 성능입니다. 공정 조건이 바뀔 때마다 재학습해야 하는 모델이라면,
> 재학습 1회가 8시간인 모델과 40분인 모델은 운영상 완전히 다른 물건입니다.
> Concept Drift가 잦은 라인일수록 이 축의 가중치를 올려야 합니다.

---
## Self-check

### Q1. RNN이 긴 시계열에서 부딪히는 두 가지 근본 한계는 무엇입니까?

<details>
<summary>정답 보기</summary>

- **장기 의존성 소실**: hidden state를 순차 갱신하며 먼 과거 정보가 점점 소실됩니다.
- **순차 연산 병목**: t 시점은 t-1이 끝나야 계산할 수 있어 병렬화가 막힙니다.
- LSTM의 게이트는 소실을 완화할 뿐이며, 병목은 구조를 바꿔야(순환 제거) 풀립니다.

</details>

---

### Q2. Attention 수식에서 √d_k로 나누는 이유와 softmax의 역할은 무엇입니까?

<details>
<summary>정답 보기</summary>

- 차원이 커지면 내적 값이 커져 softmax가 한 점에 몰립니다(기울기 소실). √d_k가 이를 완화합니다.
- softmax는 관련도 점수를 **합이 1인 확률 분포**로 바꿔 "어디를 얼마나 볼지"의 가중치로 만듭니다.
- 그 가중치로 V를 가중합한 것이 문맥이 반영된 새 표현입니다.

</details>

---

### Q3. PatchTST의 Patching이 "의미"와 "계산량" 두 문제를 동시에 푸는 원리는 무엇입니까?

<details>
<summary>정답 보기</summary>

- 점 하나는 단어만큼의 의미가 없지만, **패치(파형 조각)** 에는 지역 패턴의 의미가 담깁니다.
- 토큰 수가 L개에서 N=⌊(L−P)/S⌋+1개로 줄어 Attention 계산량 O(L²)이 O(N²)로 감소합니다.
- 즉 토큰의 의미 밀도를 올리면서 시퀀스 길이를 줄이는, 한 수로 두 문제를 건드리는 설계입니다.

</details>

---

### Q4. [현장 판단] 센서 40개짜리 설비에 시계열 모델을 올리려 합니다. Channel Independence 관점에서 무엇을 먼저 검토해야 합니까?

<details>
<summary>정답 보기</summary>

- 채널 간 스케일·파형 성격이 제각각인지 확인합니다 — 그렇다면 채널 혼합 입력은 불리합니다.
- 채널 독립 처리(가중치 공유)는 채널 수가 늘어도 모델이 커지지 않아 40채널에 유리합니다.
- 단, 채널 간 상호작용(예: 압력↔온도 인과)이 핵심인 문제라면 독립 처리가 정보를 버릴 수 있어,
  상호작용을 별도 피처나 후단 모델로 보완할지 함께 판단해야 합니다.

</details>


---
## 다음 노트북 예고 — NB04. Anomaly Transformer와 원인 추적 (XAI)

오늘 손으로 만든 Attention 행렬, 사실 예측보다 더 값진 용도가 있습니다.

> **"왜 고장이라고 판단했는가"** — NB01에서 제시한 난제 ①.

다음 노트북은 실제 수처리장 펌프 5개월 기록으로 **문제의 성격**을 먼저 확인한 뒤,
원인 센서의 정답이 있는 압출기 데이터에서 **Attention 가중치를 열어
어느 센서가 원인인지 역추적**하고 그 지목이 맞았는지까지 채점합니다.
"정상은 멀리 있는 시점과도 대화하지만, 이상은 바로 옆하고만 대화한다" —
Anomaly Transformer의 이 착상이 탐지와 설명을 한 번에 해결합니다.